# 03 — Flujo integrado: sincronizar dos escenarios

Caso de uso: **ya existen** un escenario Nacional nuevo y un escenario Regional nuevo, que
son versiones distintas del modelo. Este notebook iguala los parámetros equivalentes,
resuelve las anomalías con decisiones explícitas y regionaliza **solo** las diferencias que
deben resolverse.

El modelo regional no es el nacional con prefijos: tiene tecnologías nuevas (`TRN*`, carga
EV), fuels renombrados (`ELC` → `ELC003`), códigos que no existen en todas las regiones y
parámetros intencionalmente distintos. **El objetivo no es que sean idénticos**, sino que
los parámetros equivalentes tengan valores coherentes y que las diferencias residuales sean
intencionales y estén documentadas.

## Cómo se ejecuta (no es un *Run All* de una pasada)

| Paso | Qué hacer |
|---|---|
| 1 | Ejecutar **Secciones 0 → 2**: genera `reportes/decisiones_pendientes.xlsx` |
| 2 | **Abrir ese Excel y editar la columna `ACCION`** a mano |
| 3 | Ejecutar **Secciones 3 → 4**: aplica lo decidido y valida el resultado |

Las cuatro ACCIONes admitidas:

| ACCION | Efecto |
|---|---|
| `regionalizar` | aplicar el valor del nacional nuevo al regional (reparte por participación; los intensivos se copian) |
| `mantener_regional` | la diferencia es intencional: no se toca, queda listada como residual |
| `crear_en_regional` | falta el código en el regional — **solo alerta**, no se crea automáticamente |
| `ignorar` | no equivalente (renombramiento, código nuevo) o bajo el umbral |

**Requisito**: `notebooks/00_Generar_Mapeo.ipynb` ya ejecutado, es decir que existan los
archivos de `Insumos/Mapeo/` (`mapeo_tech_fuel.xlsx`, `diccionario_tech.xlsx`,
`diccionario_fuel.xlsx`, `participaciones.xlsx`).

El SAND regional original **nunca se modifica**: la salida es un archivo nuevo con sufijo
`_sincronizado` en `SAND_Regional/`.

## Sección 0: Configuración

In [ ]:
# --- Setup: raíz del proyecto, módulos src/ y configuración ---
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config").exists():   # ejecutado desde notebooks/
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import comparador
import regionalizador
import reporte
import sand_io
import sincronizador
import utils
import yaml_parser

utils.configurar_logging()

params_cfg = yaml_parser.cargar_params_config(RAIZ / "config" / "params_config.yaml")
paths = yaml_parser.cargar_paths_config(RAIZ / "config" / "paths_config.yaml", raiz=RAIZ)
otoole = yaml_parser.cargar_config_otoole(paths["escenario_nacional"]["config_yaml"])

print(f"Config otoole: {len(otoole['param'])} parámetros, {len(otoole['set'])} sets")
print(f"Intensivos   : {len(params_cfg['parametros_intensivos'])} | "
      f"Centinela: {params_cfg['valor_centinela']}")

In [ ]:
# --- Variables configurables de la corrida ---

# Los dos escenarios a sincronizar (versiones nuevas, ya existentes).
SAND_NAC_ESCENARIO = paths["escenario_nacional"]["sand"]
SAND_REG_ESCENARIO = paths["escenario_regional"]["sand"]

# Nacional de referencia: sirve para saber qué cambió en el nuevo escenario nacional.
# Si un combo NO cambió respecto del base, la diferencia contra el regional nació del
# lado regional y suele ser intencional. Poner None para saltarse este contraste.
SAND_NAC_BASE = RAIZ / "SAND_Nacional_base" / "01-04-2026 SAND BASE v10.xlsx"

CARPETA_MAPEO = RAIZ / "Insumos" / "Mapeo"          # salida de 00_Generar_Mapeo.ipynb
RUTA_PARTICIPACIONES = CARPETA_MAPEO / "participaciones.xlsx"

# Lista de parámetros, o 'todos_equivalentes' = los que existen en AMBOS escenarios
# y están definidos en config_depurado.yaml (los únicos comparables de raíz).
PARAMETROS_A_SINCRONIZAR = [
    "AccumulatedAnnualDemand",
    "TotalTechnologyAnnualActivityLowerLimit",
    "CapitalCost",
]

# Diferencias por debajo de este porcentaje se ignoran (ruido numérico / redondeos).
UMBRAL_DIFERENCIA_PCT = 1.0

ANIO_MAX = 2054                              # 2055 se excluye (ver CLAUDE.md)
YEARS_FILTRO = list(range(2022, ANIO_MAX + 1))

MODO_DRY_RUN = True        # True => no escribe el SAND sincronizado ni el reporte final.
                           # decisiones_pendientes.xlsx SÍ se escribe siempre: es lo que
                           # hay que revisar antes de una corrida real.

RUTA_DECISIONES = paths["outputs"]["reportes"] / "decisiones_pendientes.xlsx"
RUTA_REPORTE = paths["outputs"]["reportes"] / "Reporte_Flujo_Integrado.xlsx"
SUFIJO_SALIDA = "_sincronizado"

print(f"Nacional escenario : {SAND_NAC_ESCENARIO.name}")
print(f"Regional escenario : {SAND_REG_ESCENARIO.name}")
print(f"Nacional base      : {SAND_NAC_BASE.name if SAND_NAC_BASE else '(sin contraste)'}")
print(f"Umbral             : {UMBRAL_DIFERENCIA_PCT}% | Años: {YEARS_FILTRO[0]}–{YEARS_FILTRO[-1]}")
print(f"Dry-run            : {MODO_DRY_RUN}")

if SAND_NAC_BASE is not None and Path(SAND_NAC_BASE) == Path(SAND_NAC_ESCENARIO):
    print()
    print("AVISO: el Nacional base y el Nacional del escenario son el MISMO archivo,")
    print("       así que el contraste vs base dirá 'IGUAL' en todo y no aporta nada.")
    print("       Apuntar SAND_NAC_ESCENARIO al nacional del nuevo escenario.")

In [ ]:
# --- Carga de los dos escenarios, el mapeo y las participaciones ---
df_nac = sand_io.cargar_sand(SAND_NAC_ESCENARIO)
df_reg = sand_io.cargar_sand(SAND_REG_ESCENARIO)
print(f"Nacional: {len(df_nac):,} filas | Regional: {len(df_reg):,} filas")

# 'todos_equivalentes' => presentes en ambos escenarios y conocidos por otoole
if PARAMETROS_A_SINCRONIZAR == "todos_equivalentes":
    PARAMETROS_A_SINCRONIZAR = sorted(
        set(df_nac["Parameter"].dropna().unique())
        & set(df_reg["Parameter"].dropna().unique())
        & set(otoole["param"]))
    print(f"'todos_equivalentes' -> {len(PARAMETROS_A_SINCRONIZAR)} parámetros")
print(f"A sincronizar: {PARAMETROS_A_SINCRONIZAR}")

# Mapeo: regiones esperadas por código (0/1), renombramientos nacional->regional
# y observaciones de los diccionarios. Sin mapeo_tech_fuel.xlsx todo degrada a
# "las 7 regiones" y no hay renombramientos que explicar.
mapeo = regionalizador.cargar_mapeo_regional(CARPETA_MAPEO)
if not mapeo["disponible"]:
    display(Markdown(
        "**AVISO**: no se encontró `mapeo_tech_fuel.xlsx` — ejecutar primero "
        "`00_Generar_Mapeo.ipynb`. Sin él, las ACCIONes inferidas son mucho menos fiables."))
else:
    renames = {**mapeo["rename_tech"], **mapeo["rename_fuel"]}
    print(f"Mapeo: {len(mapeo['regiones_tech'])} tecnologías, {len(mapeo['regiones_fuel'])} fuels, "
          f"{len(renames)} renombramientos")

df_pct = regionalizador.cargar_participaciones(
    RUTA_PARTICIPACIONES,
    parametros=PARAMETROS_A_SINCRONIZAR,
    params_otoole=otoole["param"],
)
malas = regionalizador.validar_participaciones(df_pct, params_cfg["tolerancia_participacion"])
if malas.empty:
    print("Participaciones: OK, todos los grupos suman ~1.0")
else:
    display(Markdown(f"**ALERTA: {len(malas)} grupos de participación no suman 1.0** — "
                     f"lo repartido con ellos no reproducirá el total nacional"))
    display(malas.head(15))

## Sección 1: Diagnóstico de diferencias

Compara los dos escenarios nuevos con `comparador.comparar_escenarios`: ruta **aditiva**
(nacional = suma de las 7 regiones, centinelas excluidos) para demandas y límites, ruta
**intensiva** (región por región) para ratios, costos y factores.

In [ ]:
# Comparación completa de los dos escenarios nuevos
resultado_antes = comparador.comparar_escenarios(
    df_nac, df_reg, otoole["param"], params_cfg,
    modo="lista_parametros",
    parametros_filtro=PARAMETROS_A_SINCRONIZAR,
)
comp_antes = resultado_antes["comparacion"]
anomalias = resultado_antes["anomalias"]
print(f"Filas comparadas: {len(comp_antes):,} | Anomalías estructurales: {len(anomalias):,}")

In [ ]:
# --- Tabla resumen: qué parámetros están OK y cuáles hay que revisar ---
resumen_dif = sincronizador.resumen_diferencias(comp_antes, umbral_pct=UMBRAL_DIFERENCIA_PCT)

n_revisar = int((resumen_dif["Estado"] == "REVISAR").sum())
n_ok = int((resumen_dif["Estado"] == "OK").sum())
display(Markdown(
    f"### {n_revisar} parámetros con diferencias > {UMBRAL_DIFERENCIA_PCT}% · {n_ok} parámetros OK"))
display(resumen_dif)

In [ ]:
# --- Top diferencias absolutas ---
top = reporte.top_diferencias(comp_antes, n=params_cfg["top_n_grafica"])
if top.empty:
    print("Sin diferencias que rankear.")
else:
    display(Markdown("**Top diferencias absolutas (suma de |Diferencia| sobre todos los años)**"))
    display(top)
    reporte.grafica_top_diferencias(
        comp_antes, n=params_cfg["top_n_grafica"],
        titulo="Diferencias Nacional escenario vs Regional escenario")

In [ ]:
# --- Anomalías estructurales, contextualizadas con mapeo_tech_fuel.xlsx ---
# Para cada anomalía: si es un cambio de nombre conocido, si el mapeo dice que la
# tecnología no debe existir en esa región, o si hay que crearla.
mapeos_raw = comparador.cargar_mapeos(CARPETA_MAPEO)

if anomalias.empty:
    print("Sin anomalías estructurales.")
else:
    display(Markdown("**Anomalías por tipo**"))
    display(anomalias["Tipo_Anomalia"].value_counts().rename("Casos").to_frame())

    if mapeos_raw["mapeo"] is not None:
        ctx = comparador.contextualizar_anomalias(anomalias, mapeos_raw["mapeo"],
                                                  regiones=list(params_cfg["prefijo_region"].values()))
        display(Markdown("**Acción sugerida por el mapeo** "
                         "(`cambio_de_nombre` = falso positivo · `no_existe_en_region` = esperado · "
                         "`verificar_creacion` = falta de verdad)"))
        display(ctx["ACCION_SUGERIDA"].value_counts().rename("Casos").to_frame())
        display(ctx.head(20))
    else:
        display(Markdown("**Sin `mapeo_tech_fuel.xlsx`**: las anomalías no se pueden contextualizar."))
        display(anomalias.head(20))

In [ ]:
# --- ¿Qué cambió en el Nacional nuevo respecto del Nacional base? ---
# Desempata al decidir: CAMBIADO/NUEVO => la diferencia viene del escenario nacional
# (regionalizar suele ser lo correcto); IGUAL => nació del lado regional (probablemente
# intencional). Es solo información: no fija ninguna ACCION.
if SAND_NAC_BASE is not None and Path(SAND_NAC_BASE).exists():
    df_nac_base = sand_io.cargar_sand(SAND_NAC_BASE)
    cambios_base = sincronizador.cambios_vs_base(
        df_nac, df_nac_base,
        parametros=PARAMETROS_A_SINCRONIZAR,
        hasta_anio=ANIO_MAX,
        tolerancia=params_cfg["tolerancia_comparacion"],
    )
    display(Markdown("**Cambios del Nacional nuevo vs Nacional base**"))
    display(cambios_base["Estado_Vs_Base"].value_counts().rename("Combos").to_frame())
else:
    cambios_base = pd.DataFrame()
    print("Sin SAND_NAC_BASE: se omite el contraste contra la referencia base.")

## Sección 2: Clasificación de diferencias

Colapsa el diagnóstico a **una fila por (Parámetro, TECHNOLOGY, FUEL)** con su
`TIPO_DIFERENCIA` y una `ACCION` **inferida** desde el mapeo. Reglas de la inferencia:

| Situación detectada | ACCION propuesta |
|---|---|
| El mapeo declara un renombramiento (`ELC` → `ELC003`) | `ignorar` — la diferencia es de nomenclatura |
| El mapeo dice que el código no va en ninguna región | `mantener_regional` |
| Falta en regiones que el mapeo sí espera | `crear_en_regional` |
| Existe en ambos y el valor difiere (con participación, o intensivo) | `regionalizar` |
| Existe en ambos y el valor difiere, pero sin participación | `mantener_regional` + *REVISAR a mano* |
| Sin información en el mapeo | `mantener_regional` + *REVISAR a mano* |

La propuesta es un punto de partida, **no una decisión**: la columna `Motivo_Accion` dice
por qué se propuso cada una, para poder discutirla.

In [ ]:
decisiones = sincronizador.construir_decisiones(
    comp_antes, anomalias, df_nac, df_reg,
    mapeo=mapeo, cfg=params_cfg,
    umbral_pct=UMBRAL_DIFERENCIA_PCT,
    df_pct=df_pct,
)

# Contexto extra (informativo): qué cambió en el nacional respecto del base
if not cambios_base.empty:
    decisiones = sincronizador.anotar_cambios_vs_base(decisiones, cambios_base)

if decisiones.empty:
    display(Markdown("### Sin diferencias sobre el umbral ni anomalías: nada que decidir."))
else:
    display(Markdown(f"### {len(decisiones)} diferencias a clasificar"))
    display(pd.crosstab(decisiones["TIPO_DIFERENCIA"], decisiones["ACCION"], margins=True))
    display(Markdown("**Las que se aplicarían tal cual están** (`regionalizar`)"))
    display(decisiones[decisiones["ACCION"] == sincronizador.ACCION_REGIONALIZAR].head(20))
    display(Markdown("**Marcadas para revisar a mano** (motivo con *REVISAR*)"))
    display(decisiones[decisiones["Motivo_Accion"].str.contains("REVISAR", na=False)].head(20))

In [ ]:
# El Excel de decisiones se escribe SIEMPRE, también en dry-run: es el artefacto
# que hay que revisar antes de aplicar nada.
if not decisiones.empty:
    sincronizador.exportar_decisiones(decisiones, RUTA_DECISIONES)
    display(Markdown(
        f"### Editar ahora: `{RUTA_DECISIONES}`\n\n"
        f"Abrir el archivo, ajustar la columna **ACCION** de las {len(decisiones)} filas "
        f"(valores válidos en la hoja *Instrucciones*), guardar, y **continuar en la "
        f"celda siguiente** — no hace falta re-ejecutar las secciones 0–2."))

### 2b. Releer las decisiones editadas

**Ejecutar esta celda después de guardar el Excel.** Revalida la columna `ACCION`: una
acción no reconocida aborta aquí y no más adelante, cuando ya se habría escrito algo.

In [ ]:
decisiones_finales = sincronizador.leer_decisiones(RUTA_DECISIONES, estricto=True)

display(Markdown(f"**{len(decisiones_finales)} decisiones releídas**"))
display(decisiones_finales["ACCION"].value_counts().rename("Casos").to_frame())

# Qué cambió respecto de lo que el notebook propuso
if len(decisiones_finales) == len(decisiones):
    editadas = (decisiones_finales["ACCION"].values != decisiones["ACCION"].values).sum()
    print(f"Acciones modificadas a mano: {editadas} de {len(decisiones)}")

## Sección 3: Aplicar decisiones

- `regionalizar` → `regionalizador.regionalizar`, acotado a los parámetros y códigos
  decididos. Los renombramientos del mapeo se aplican al construir el código regional
  (`ELC` nacional se escribe `AN_ELC003`, no `AN_ELC`).
- `crear_en_regional` → **solo alerta**, con la lista exacta de qué crear y en qué regiones.
  La creación requiere intervención manual: hay que definir tecnología, fuels de entrada y
  salida, y participaciones.
- `mantener_regional` / `ignorar` → no hacen nada.

In [ ]:
aplicado = sincronizador.aplicar_decisiones(
    df_nac, df_reg, df_pct, decisiones_finales,
    params_otoole=otoole["param"], cfg=params_cfg, mapeo=mapeo,
    years_filtro=YEARS_FILTRO,
)
sands = aplicado["sands"]

display(Markdown("**Resumen por parámetro**"))
display(aplicado["resumen"])

log = aplicado["log"]
if not log.empty:
    omitidos = log[log["Nivel"] == "OMITIDO"]
    if not omitidos.empty:
        display(Markdown(f"**{len(omitidos)} combos marcados `regionalizar` que aun así se "
                         f"omitieron** (sin correspondencia regional o sin participación)"))
        display(omitidos.head(20))

In [ ]:
# --- Alerta: qué falta crear en el regional (NO se crea automáticamente) ---
pendientes = aplicado["pendientes_creacion"]

if pendientes.empty:
    print("Nada marcado como 'crear_en_regional'.")
else:
    display(Markdown(
        f"### ATENCIÓN: {len(pendientes)} códigos a crear manualmente en el escenario regional\n\n"
        f"Este notebook **no** los crea: definirlos requiere decidir tecnología, fuels de "
        f"entrada/salida y participaciones. Lista exacta de qué crear y dónde:"))
    display(pendientes)
    for _, f in pendientes.iterrows():
        codigo = f["TECHNOLOGY"] if pd.notna(f["TECHNOLOGY"]) else f["FUEL"]
        regiones = f.get("Regiones_Faltantes") or "(revisar mapeo)"
        print(f"  {f['Parametro']:<45} {codigo:<22} -> crear en: {regiones}")

In [ ]:
# --- SAND regional sincronizado (archivo nuevo: el original no se toca) ---
df_reg_sinc = sincronizador.integrar_sands(df_reg, sands)

ruta_salida = SAND_REG_ESCENARIO.parent / f"{SAND_REG_ESCENARIO.stem}{SUFIJO_SALIDA}.xlsx"
filas_sustituidas = sum(len(d) for d in sands.values())
print(f"Regional original    : {len(df_reg):,} filas")
print(f"Regional sincronizado: {len(df_reg_sinc):,} filas ({filas_sustituidas:,} sustituidas)")

if MODO_DRY_RUN:
    display(Markdown(f"**DRY-RUN — no se escribió nada.** Se habría generado `{ruta_salida}`"))
else:
    sand_io.escribir_sand(df_reg_sinc, ruta_salida)
    print(f"SAND escrito: {ruta_salida}")
    ruta_log, ruta_log_txt = reporte.escribir_log_regionalizacion(
        aplicado["log"], paths["outputs"]["reportes"], nombre_base="log_flujo_integrado")
    print(f"Log: {ruta_log}")

## Sección 4: Validación post-sincronización

Vuelve a comparar, ahora contra el regional sincronizado (el de memoria, exista o no el
archivo). Estados por combinación:

- **RESUELTA** — estaba sobre el umbral y ya no lo está.
- **PERSISTE** — sigue sobre el umbral. Con `ACCION = mantener_regional` es una
  **diferencia residual intencional**, documentada por su `Motivo_Accion`.
- **NUEVA** — no estaba antes y apareció después: señal de que la sincronización rompió
  algo. Deberían ser cero.

In [ ]:
resultado_despues = comparador.comparar_escenarios(
    df_nac, df_reg_sinc, otoole["param"], params_cfg,
    modo="lista_parametros",
    parametros_filtro=PARAMETROS_A_SINCRONIZAR,
)
comp_despues = resultado_despues["comparacion"]

resolucion = sincronizador.comparar_resolucion(
    comp_antes, comp_despues, decisiones_finales, umbral_pct=UMBRAL_DIFERENCIA_PCT)
m = resolucion["metricas"]

display(Markdown(f"""
### Resultado

| | Combinaciones |
|---|---|
| Sobre el umbral **antes** | {m['combos_antes']:,} |
| Sobre el umbral **después** | {m['combos_despues']:,} |
| **Resueltas** | {m['resueltas']:,} |
| Persisten | {m['persisten']:,} |
| **Nuevas** (deberían ser 0) | {m['nuevas']:,} |
| De las que persisten, intencionales (`mantener_regional`) | {m['residuales_intencionales']:,} |
"""))

if m["nuevas"] > 0:
    display(Markdown("**ALERTA: la sincronización introdujo diferencias que no existían.** "
                     "Revisar estas combinaciones antes de usar el SAND sincronizado:"))
    display(resolucion["detalle"].query("Estado == 'NUEVA'").head(20))

In [ ]:
# --- Diferencias residuales: las que quedan, y por qué ---
residuales = resolucion["residuales"]
sin_decidir = resolucion["detalle"].query("Estado == 'PERSISTE' and ACCION == '(sin decisión)'")

if residuales.empty:
    print("Sin diferencias residuales intencionales.")
else:
    display(Markdown(f"**{len(residuales)} diferencias residuales intencionales** "
                     f"(marcadas `mantener_regional`, con su motivo documentado)"))
    display(residuales.head(30))

if not sin_decidir.empty:
    display(Markdown(f"**{len(sin_decidir)} diferencias persisten SIN decisión asociada** — "
                     f"quedaron fuera de la tabla (bajo el umbral al construirla) pero siguen "
                     f"sobre el umbral. Revisar:"))
    display(sin_decidir.head(20))

In [ ]:
# --- Reporte final ---
hojas = {
    "Antes": sincronizador.combos_sobre_umbral(comp_antes, UMBRAL_DIFERENCIA_PCT),
    "Despues": sincronizador.combos_sobre_umbral(comp_despues, UMBRAL_DIFERENCIA_PCT),
    "Decisiones": decisiones_finales,
    "Diferencias_Residuales": residuales,
    "Resolucion_Detalle": resolucion["detalle"],
    "Resumen_Parametros": resumen_dif,
    "Anomalias": anomalias,
    "Pendientes_Creacion": pendientes,
}

if MODO_DRY_RUN:
    display(Markdown(f"**DRY-RUN — no se escribió el reporte.** Se habría generado "
                     f"`{RUTA_REPORTE}` con las hojas:"))
    for nombre, df in hojas.items():
        estado = f"{len(df):,} filas" if df is not None and not df.empty else "(vacía, se omite)"
        print(f"  {nombre:<24} {estado}")
else:
    reporte.exportar_excel(RUTA_REPORTE, hojas)
    print(f"Reporte final: {RUTA_REPORTE}")

## Siguiente paso

1. Resolver a mano los `crear_en_regional` de la Sección 3 y volver a correr el flujo.
2. Con `MODO_DRY_RUN = False`, integrar el SAND sincronizado y convertirlo con otoole:

```bash
otoole convert csv excel CSV_Regional salida.xlsx config_depurado.yaml
```

*Smoke tests de los módulos: `python tests/test_smoke.py` desde la raíz del proyecto.*